# detach() 是"截断梯度的剪刀"

当你调用 `.detach()`，等于说：**"从这里开始，后面的计算跟我前面没关系了"**。

新变量和原变量数值一样，但它"忘记"了自己是怎么被算出来的。反向传播传到它这里就停了，不会再往前传。

**场景举例**：你有一个预训练好的模型 A，想用它的输出喂给模型 B 训练，但**不想让梯度传回去更新模型 A**。那就 `.detach()` 一下。

In [2]:
import torch

In [3]:
x = torch.tensor(2.0, requires_grad=True)
y = x.detach()

print(x)
print(y)

tensor(2., requires_grad=True)
tensor(2.)


y 虽然经过了detach操作，不再存储梯度，和x为不同的对象，但是和x仍然共享数据

In [ ]:
# 检查是否为相同的对象
print(id(x))
print(id(y))

5643438016
5643444336


In [5]:
# 检查底层的数据
print(x.untyped_storage().data_ptr())
print(y.untyped_storage().data_ptr())

5492102592
5492102592


In [6]:
# 分别对x 和 y做后续的运算
z1 = x ** 2
z2 = y ** 2
print(z1)
print(z2)

tensor(4., grad_fn=<PowBackward0>)
tensor(4.)


In [ ]:
z1.sum().backward()
# detach操作后无法反向传播
z2.sum().backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### 一个应用detach的具体例子


In [ ]:
x = torch.ones(2,2,requires_grad=True)
# y = x²，计算图: x → y (MulBackward0)，梯度可以从 y 传回 x
y = x * x
print(x)
print(y)

In [ ]:
# u = y.detach()，u 的数值和 y 一样，但 u 被从计算图中"剪断"了
# u 没有 grad_fn，反向传播到 u 就停了，不会再传到 y 和 x
u = y.detach()
print(u)

In [ ]:
# ========================================
# 关键：z = u * x，其中 u = y.detach() 是常量（无梯度）
# ========================================
#
# 如果不用 detach（直接用 y * x）：
#   z = y * x = x² * x = x³
#   ∂z/∂x = 3x² = [[3, 3], [3, 3]]
#
# 用了 detach 之后：
#   z = u * x，u 被当作常量（不参与梯度计算）
#   ∂z/∂x = u = x² = [[1, 1], [1, 1]]
#
# 对比：x.grad 从 [[3,3],[3,3]] 变成了 [[1,1],[1,1]]
# 这就是 detach 的作用——截断了从 u 回传到 y 再到 x 的梯度路径
z = u * x
z.sum().backward()
print(x.grad)  # [[1,1],[1,1]]，而不是 [[3,3],[3,3]]